# 09 · Inferencia — del modelo registrado a una función que cualquiera puede usar

> **BORRADOR para revisión del grupo.** Guía: `material-del-curso/kit/guia-validacion-registro-y-app.md`, secciones 6 a 8.

Un modelo entrenado no es un producto. Un producto recibe datos **sucios o incompletos de una persona real** y responde con algo útil o con un **error claro**. Esta etapa construye ese puente:

1. **Exportar** el modelo `champion` a un archivo que viaja con la app.
2. Escribir el **esquema de entrada**: qué columnas, qué rangos son posibles, qué categorías.
3. Probar la función de predicción con datos buenos, incompletos y **imposibles**.
4. Comprobar que el archivo exportado **es** el modelo que se evaluó.
5. Completar la **ficha del modelo**.

## 0. Datos y rutas

In [1]:
import json
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
ROOT = ROOT if (ROOT / "src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import joblib
import numpy as np
import pandas as pd

from proyecto.inference.serving import RAW_COLUMNS, ServingModel, build_defaults, to_raw
from proyecto.models.metrics import COST_FALSE_NEGATIVE, COST_FALSE_POSITIVE

train = pd.read_parquet(ROOT / "data" / "processed" / "train.parquet")
raw_train = to_raw(train)                       # lo que escribiría una persona: "Yes"/"No", "< 1 Year", etc.

# === Configuración ===
TARGET = "Response"
NUMERICAS = ["Age", "Annual_Premium", "Region_Code"]
CATEGORICAS = ["Gender", "Vehicle_Age", "Vehicle_Damage", "Previously_Insured"]
CLASES = ["no interesado", "interesado"]                      # en el orden de predict_proba
MATRIZ_DE_COSTOS = [[0.0, COST_FALSE_POSITIVE],                # filas = clase real, columnas = clase decidida
                    [COST_FALSE_NEGATIVE, 0.0]]                # perder a un interesado cuesta 5 veces una llamada inútil (R = 5)
RANGOS_POSIBLES = {"Age": (18, 100), "Annual_Premium": (1, 10_000_000)}   # lo POSIBLE en el mundo real, no lo observado

assert sorted(NUMERICAS + CATEGORICAS) == sorted(RAW_COLUMNS)
print(f"entrenamiento: {len(train):,} clientes; el formulario pedirá {len(RAW_COLUMNS)} datos: {RAW_COLUMNS}")

entrenamiento: 304,887 clientes; el formulario pedirá 7 datos: ['Age', 'Annual_Premium', 'Region_Code', 'Gender', 'Vehicle_Age', 'Vehicle_Damage', 'Previously_Insured']


## 1. Exportar el modelo `champion`

La fuente de verdad es el **registro**. Si no hay conexión, se usa la copia local de la etapa 7 y el notebook lo dice, para que nadie crea que usa una cosa y use otra.

In [2]:
import mlflow
from proyecto.tracking.mlflow_utils import configure_mlflow

pipeline = None
try:
    registro = json.loads((ROOT / "reports" / "08_registro.json").read_text(encoding="utf-8"))
    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always")
        donde = configure_mlflow()
    pipeline = mlflow.sklearn.load_model(f"models:/{registro['modelo_registrado']}@champion")
    origen = f"registro de MLflow ({donde}), {registro['modelo_registrado']}@champion (versión {registro['version']})"
except Exception as error:
    print("No se pudo leer el registro:", str(error).splitlines()[0][:90])

if pipeline is None:
    pipeline = joblib.load(ROOT / "models" / "final_pipeline.joblib")
    origen = "copia local de la etapa 7 (el registro no respondió)"

defaults = build_defaults(raw_train)                  # valores neutros para los "no sé", solo del entrenamiento
modelo = ServingModel(pipeline, defaults)
joblib.dump(modelo, ROOT / "models" / "model.joblib")
print("modelo exportado a models/model.joblib desde:", origen)
print("valores usados cuando alguien marca «no sé»:", defaults)

2026/11/05 19:36:15 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at c:\Users\Usuario\Downloads\Grupo2\.venv Proyecto Gruo2\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


modelo exportado a models/model.joblib desde: registro de MLflow (databricks (token)), workspace.default.venta_cruzada_seguros@champion (versión 1)
valores usados cuando alguien marca «no sé»: {'Age': 36.0, 'Annual_Premium': 31697.0, 'Region_Code': 28.0, 'Gender': 'Male', 'Vehicle_Age': '1-2 Year', 'Vehicle_Damage': 'Yes', 'Previously_Insured': 'No'}


## 2. El esquema de entrada

La app y la función de predicción necesitan saber qué se puede preguntar: qué columnas, qué rango es **posible** y qué categorías conoce el modelo. Se calcula una vez con el entrenamiento y se guarda como un archivo que viaja con el modelo (`models/input_schema.json`).

> `RANGOS_POSIBLES` es **conocimiento del grupo**, no de los datos: una edad de 200 es imposible aunque nunca haya aparecido. Fuera de ese rango, la predicción **se niega** a responder.

In [3]:
from proyecto.inference.schema import build_schema, save_schema

LABELS = {"Age": "Edad (años)", "Annual_Premium": "Prima anual del seguro de salud", "Region_Code": "Código de región",
          "Gender": "Género", "Vehicle_Age": "Antigüedad del vehículo", "Vehicle_Damage": "¿El vehículo ha tenido daños?",
          "Previously_Insured": "¿Ya tiene seguro de vehículo?"}
schema = build_schema(raw_train, NUMERICAS, CATEGORICAS, CLASES, cost_matrix=MATRIZ_DE_COSTOS, valid_range=RANGOS_POSIBLES, labels=LABELS,
                      review_threshold=0.0)
schema["app_title"] = "¿A qué clientes conviene llamar? Venta cruzada de seguro de vehículo"
schema["decision_labels"] = {"interesado": "LLAMAR", "no interesado": "NO LLAMAR"}
schema["responsible_use"] = (
    "- Esta herramienta **ordena a quién llamar primero**; no decide si se vende o se niega un seguro.\n"
    "- Usa la **edad y el género** del cliente. Legal debe aprobar su uso; el modelo casi no llama a menores de 26 ni a mayores de 65.\n"
    "- Entrenada con datos históricos de una sola aseguradora: no la uses con otra población sin validarla de nuevo.\n"
    "- El costo asume que perder un comprador cuesta 5 veces una llamada inútil (hipótesis por confirmar).\n"
    "- Revisa los **límites del modelo** en `models/ficha_modelo.md`."
)
save_schema(schema, ROOT / "models" / "input_schema.json")
print(f"esquema guardado: {len(NUMERICAS)} numéricas, {len(CATEGORICAS)} categóricas, clases {CLASES}")
print("categorías conocidas:", schema["categories"])

esquema guardado: 3 numéricas, 4 categóricas, clases ['no interesado', 'interesado']
categorías conocidas: {'Gender': ['Female', 'Male'], 'Vehicle_Age': ['1-2 Year', '< 1 Year', '> 2 Years'], 'Vehicle_Damage': ['No', 'Yes'], 'Previously_Insured': ['No', 'Yes']}


## 3. Probar la predicción

Tres casos que importan: una fila normal, una **incompleta** (el usuario marcó "no sé") y una con un dato **imposible**. Qué se espera: la primera responde; la segunda también (se usa un valor neutro); la tercera **se niega y explica por qué**.

In [4]:
from proyecto.inference.predict import InputValidationError, predict_with_scores

modelo = joblib.load(ROOT / "models" / "model.joblib")
normal = pd.DataFrame([{"Age": 34, "Annual_Premium": 31000.0, "Region_Code": 28, "Gender": "Male", "Vehicle_Age": "1-2 Year",
                        "Vehicle_Damage": "Yes", "Previously_Insured": "No"}])
print("— fila normal (hombre de 34 años, sin seguro previo, con daño) —")
print(predict_with_scores(normal, modelo, schema).round(3).to_string())

contraste = normal.assign(Vehicle_Damage="No", Previously_Insured="Yes")
print("\n— el mismo cliente con seguro previo y sin daño —")
print(predict_with_scores(contraste, modelo, schema).round(3).to_string())

incompleta = normal.copy()
incompleta["Age"] = float("nan")
incompleta["Gender"] = np.nan
print("\n— con la edad y el género en «no sé» —")
print(predict_with_scores(incompleta, modelo, schema).round(3).to_string())

imposible = normal.copy()
imposible["Age"] = 200
try:
    predict_with_scores(imposible, modelo, schema)
    print("\nERROR: debía negarse a predecir con una edad de 200 años")
except InputValidationError as error:
    print("\n— dato imposible —\nError claro:", error)

categoria_rara = normal.assign(Vehicle_Age="3 años")
try:
    predict_with_scores(categoria_rara, modelo, schema)
except Exception as error:
    print("\n— categoría desconocida —\nSe rechaza con:", type(error).__name__, "-", str(error)[:90])

— fila normal (hombre de 34 años, sin seguro previo, con daño) —


     decision  proba_no interesado  proba_interesado    most_likely  expected_cost  needs_review
0  interesado                0.607             0.393  no interesado          0.607          True

— el mismo cliente con seguro previo y sin daño —
        decision  proba_no interesado  proba_interesado    most_likely  expected_cost  needs_review
0  no interesado                0.999             0.001  no interesado          0.005         False

— con la edad y el género en «no sé» —


     decision  proba_no interesado  proba_interesado    most_likely  expected_cost  needs_review
0  interesado                0.644             0.356  no interesado          0.644          True

— dato imposible —
Error claro: Age: debe estar entre 18 y 100

— categoría desconocida —
Se rechaza con: InputValidationError - Vehicle_Age: valor desconocido ['3 años']; se esperaba uno de ['1-2 Year', '< 1 Year', '> 


## 4. Comprobar que el archivo exportado es el modelo que se evaluó

Antes de dárselo a la app: `models/model.joblib` debe dar **las mismas métricas** que se midieron en la prueba (etapa 7). Si no coinciden, se estaría entregando un modelo distinto al que se evaluó. **Esta comprobación no cambia nada del modelo ni se usa para decidir**: solo verifica que es el mismo.

In [5]:
from proyecto.models.tuning import per_client_cost

info = json.loads((ROOT / "reports" / "07_metricas_test.json").read_text(encoding="utf-8"))
test = pd.read_parquet(ROOT / "data" / "processed" / "holdout.parquet")
proba_test = modelo.predict_proba(to_raw(test))[:, 1]
costo_exportado = float(per_client_cost(test[TARGET], proba_test).mean())
costo_evaluado = info["prueba"]["metricas"]["modelo final (LightGBM)"]["cost"]
print(f"costo por cliente en la prueba: exportado {costo_exportado:.6f} | evaluado en la etapa 7 {costo_evaluado:.6f}")
assert abs(costo_exportado - costo_evaluado) < 1e-9, "el modelo exportado no es el que se evaluó"
print("el modelo exportado es el que se evaluó")

costo por cliente en la prueba: exportado 0.331702 | evaluado en la etapa 7 0.331702
el modelo exportado es el que se evaluó


## 5. La ficha del modelo

Un documento corto que cualquiera pueda leer antes de usar el modelo: qué es, con qué datos, qué rendimiento tiene y **qué no hay que hacer con él**. Está en `models/ficha_modelo.md`, completada con los resultados de la prueba.

In [6]:
ficha = ROOT / "models" / "ficha_modelo.md"
texto = ficha.read_text(encoding="utf-8")
pendientes = texto.count("_______________")
print(f"ficha_modelo.md: {pendientes} campos en blanco" if pendientes else "ficha_modelo.md completa")

ficha_modelo.md completa


## 6. Siguiente: la app

Con `models/model.joblib` y `models/input_schema.json` en su sitio, la app de la plantilla ya funciona:

```bash
streamlit run app/streamlit_app.py
```

Comprueba que: el formulario aparece, una predicción normal responde, **"No sé" no rompe nada** y un dato imposible muestra un error claro. El despliegue en internet es la clase 5.